## 1. Install Dependencies

In [ ]:
!pip install transformers torchaudio sentence-transformers spacy
!python -m spacy download en_core_web_sm
```

## 2. Import Required Libraries

In [ ]:
import os
import torch
import torchaudio
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score
```

## 3. Transcribe Audio to Text

In [ ]:
def transcribe_audio(file_path):
    processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
    model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")
    waveform, sample_rate = torchaudio.load(file_path)
    if sample_rate != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
        waveform = resampler(waveform)
    inputs = processor(waveform.squeeze().numpy(), return_tensors="pt", sampling_rate=16000)
    with torch.no_grad():
        logits = model(**inputs).logits
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor.batch_decode(predicted_ids)[0]
    return transcription
```

## 4. LSTM with Attention Model

In [ ]:
class LSTMAttentionModel(nn.Module):
    def __init__(self, embedding_dim, hidden_dim):
        super(LSTMAttentionModel, self).__init__()
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.attention = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attn_weights = torch.softmax(self.attention(lstm_out), dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        return context, attn_weights

class GrammarScoringModel(nn.Module):
    def __init__(self, embedding_dim, hidden_dim):
        super(GrammarScoringModel, self).__init__()
        self.encoder = LSTMAttentionModel(embedding_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        context, _ = self.encoder(x)
        return self.fc(context)
```

## 5. Sentence Embedding & Data Preparation

In [ ]:
sbert_model = SentenceTransformer("paraphrase-MiniLM-L6-v2")

def get_embeddings(text):
    sentences = text.split(". ")
    embeddings = sbert_model.encode(sentences, convert_to_tensor=True)
    return embeddings

def prepare_sample(text):
    embeddings = get_embeddings(text)
    return embeddings.unsqueeze(0)  # Add batch dimension
```

## 6. Load Trained Model or Train on Sample Data

In [ ]:
embedding_dim = 384
hidden_dim = 140

model = GrammarScoringModel(embedding_dim, hidden_dim)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Optional: Load trained weights
# model.load_state_dict(torch.load("grammar_model.pt"))
```

## 7. Score Prediction Function

In [ ]:
def predict_grammar_score(audio_path):
    text = transcribe_audio(audio_path)
    embeddings = prepare_sample(text).to(device)
    model.eval()
    with torch.no_grad():
        score = model(embeddings)
    return text, score.item()
```

## 8. Run Prediction

In [ ]:
audio_path = "hello.wav"
text, score = predict_grammar_score(audio_path)
print(f"Transcribed Text:\n{text}\n")
print(f"Predicted Grammar Score: {score:.2f}")